# The walk-forward engine: what each month may see

*A learning exercise performed in role: a simulated mandate with no client and no institution. Nothing in this notebook is investment advice, a recommendation, or a client communication.*

**Entry point** `python3 -m portfolio_workbench.evaluate.walkforward`

**Modules covered** `evaluate/walkforward.py`

_Generated from the code by `python3 -m reporting.notebooks`: the module headers below are read out of the modules themselves, and the run is the entry point's own output._

## 1. What this module does, and the source of every method in it

The walk-forward engine decides, month by month, which observations a decision may use, and records the decision so that the boundary can be asserted rather than trusted. The entry point prints the step table with each window's span, its availability date and what gated it.

**Sources.** Every public function of the modules this notebook covers, and what it traces to. The map is checked over the code by the acceptance fixture, so a method added without a source fails a command rather than going unnoticed.

- `evaluate/walkforward.py::assert_no_look_ahead` traces to the out-of-sample protocol of this effort: rolling sixty-month estimation, monthly refit, and an assertion that every estimation window ends before the month it trades
- `evaluate/walkforward.py::block` traces to the out-of-sample protocol of this effort: rolling sixty-month estimation, monthly refit, and an assertion that every estimation window ends before the month it trades
- `evaluate/walkforward.py::frame` traces to the out-of-sample protocol of this effort: rolling sixty-month estimation, monthly refit, and an assertion that every estimation window ends before the month it trades
- `evaluate/walkforward.py::main` traces to the out-of-sample protocol of this effort: rolling sixty-month estimation, monthly refit, and an assertion that every estimation window ends before the month it trades
- `evaluate/walkforward.py::steps` traces to the out-of-sample protocol of this effort: rolling sixty-month estimation, monthly refit, and an assertion that every estimation window ends before the month it trades

## 2. Why it works this way, including what was rejected

_The module headers, verbatim: each records why the module is shaped the way it is, what was rejected, and the measurement that settled it. They are quoted here rather than restated, so the notebook cannot drift from the code._

**`evaluate/walkforward.py`**

One engine decides, for every month, which data was available and which model was fitted.

The boundary the whole comparison rests on is one inequality: an estimate formed at the close of a
month cannot read the next month's bar. It is a property of the calendar the windows are cut on, and
it is invisible in a weight path - a cell that looked one month ahead returns plausible weights and a
flattering return series, and nothing in the row would say so. So the engine keeps a **record** rather
than an intention. Every step carries the month traded, the window estimated on, the moment that
window's last bar became readable and which bar set that moment; the run is then checked against the
record, rather than each step being trusted as it goes.

Four things here are decisions rather than mechanics.

**The window ends the month before the month it trades, and nothing else will do.** A window that
stopped a month earlier would be conservative rather than wrong, and it would silently shorten every
estimate in the package; a window that reached into the traded month is the leakage the design exists
to prevent. The engine asserts equality, so neither can happen quietly.

**The estimate is the window's own, the count included.** The number of components a factor
covariance retains is decided inside the trailing window, so the engine carries the count beside the
step it belongs to: the count a step reports is the one its own window decided, and the row says which
window that was. A model fitted on a longer window than the step trades behind would be leakage
whether or not its count differed.

**The availability that gated a step is the traded month's own first day.** A monthly bar labelled
with the month's first day carries that month's last close, so it is readable from the first day of
the following month; the window's last bar therefore becomes readable exactly as the traded month
opens. The rule is written once in the data layer and read here, and the record states the moment it
produced rather than the argument for it.

**The record is what the boundary is checked against, and it is checked on the whole run.** The
assertion is a function over the steps rather than a flag inside the loop, which is what lets a
planted break be caught: a window that reaches its own traded month, a window that stops short of the
last bar the month could have read, an availability that does not follow from the bar it names, a
window that is not the contiguous run it claims, or two counts on one window each fail here without a
weight path having been built first.

## 3. The data contract it consumes, and the as-of rule

Rolling sixty-month estimation with a monthly refit: a step trading month m is estimated on the sixty months ending at m-1, and every bar enters only from the first day of the month after the month it describes. The expanding variant trades the same months with a window that grows from the start of the panel. Every window's last month precedes every month it trades, and the assertion that checks it is part of the run rather than a comment.

## 4. The worked example on small numbers, with the identity checked

The boundary is the whole claim, so the worked example builds the steps for a short panel and asserts it directly: the window ends the month before the month it trades, and the traded months are the panel's own tail.

The cell below runs on numbers small enough to check by hand and asserts the identity, so a reader can see the arithmetic rather than take the module's word for it.

In [1]:
import pandas as pd

from portfolio_workbench.evaluate import walkforward

months = pd.period_range("2020-01", periods=70, freq="M")
steps = walkforward.steps(months, kind="rolling", window=60)

assert len(steps) == 10
assert steps[0]["traded"] == pd.Period("2025-01", freq="M")
assert steps[0]["window_end"] == pd.Period("2024-12", freq="M")
for step in steps:
    assert step["window_end"] < step["traded"], step
    assert step["available_from"] > step["window_end"].to_timestamp(how="end")
print(f"{len(steps)} steps, first trading {steps[0]['traded']} on a window ending {steps[0]['window_end']}")

10 steps, first trading 2025-01 on a window ending 2024-12


## 5. The real run: inputs, parameters, provenance block

The provenance block is printed first, then the parameters this module decides under, then the entry point's own report. The report is the module's output rather than a transcription of it, so a number quoted from a notebook is the number the module prints.

In [2]:
from portfolio_workbench.data import loader, universe

document = loader.load_panel()
months = document["months"]
print(f"snapshot {document['snapshot_id']}, taken as of {document['as_of']}")
print(f"panel {len(months)} months {months.min()}..{months.max()} across {len(universe.TICKERS)} sleeves")
print("manifest fields: " + ", ".join(sorted(document["manifest"])))

from portfolio_workbench.evaluate import walkforward

from portfolio_workbench.factors import exposures

print(f"protocols {tuple(walkforward.PROTOCOLS)}, refit {walkforward.REFIT}")
print(f"estimation window {exposures.WINDOW} months, out-of-sample months only")

snapshot 2026-09-13, taken as of 2026-09-13T07:56:28+00:00
panel 191 months 2010-09..2026-07 across 11 sleeves
manifest fields: created, excluded, files, instruments, snapshot_id, window
protocols ('rolling', 'expanding'), refit monthly
estimation window 60 months, out-of-sample months only


In [3]:
import subprocess
import sys

finished = subprocess.run(
    [sys.executable, "-m", "portfolio_workbench.evaluate.walkforward"], capture_output=True, text=True, cwd="."
)
print(finished.stdout)
assert finished.returncode == 0, finished.stderr

[table] the expanding protocol: 131 steps 2015-09..2026-07 over 131 windows, 16244 sleeve-months read
[table] 2015-09  window 2010-09..2015-08   60 months,  59 observations  available 2015-09-01 from the 2015-08 bar  refit monthly
[table] 2026-07  window 2010-09..2026-06  190 months, 189 observations  available 2026-07-01 from the 2026-06 bar  refit monthly
[table] the expanding window runs 60 months behind 2015-09 and 190 months behind 2026-07, trading the same months as the rolling one
[table] every window's last bar becomes readable on the first day of the month it is traded
[table] the rolling protocol: 131 steps 2015-09..2026-07 over 131 windows, 7859 sleeve-months read
[table] 2015-09  window 2010-09..2015-08   60 months,  59 observations  available 2015-09-01 from the 2015-08 bar  refit monthly
[table] 2015-10  window 2010-10..2015-09   60 months,  60 observations  available 2015-10-01 from the 2015-09 bar  refit monthly
[table] 2015-11  window 2010-11..2015-10   60 months,  60 

## 6. Results, and how to read them, including the resolution limit and what a reader must not conclude

The engine's output is a decision about what a month may see, and its resolution limit is the panel: 131 traded months is what the sixty-month requirement leaves of 191. The planted break in the acceptance fixture is the evidence that the assertion fires rather than merely existing. A reader must not read the first window's shorter observation count as a data gap: the panel simply starts there, and the step records it.

## 7. What this module does not establish

Nothing here establishes that the protocol matches how a manager would actually trade. It fixes a monthly decision with a monthly refit and no intra-month action, and it does not model the delay between a decision and its execution beyond the one-month availability rule.